In [ ]:
import sys
!{sys.executable} -m pip install pandas matplotlib numpy seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import os
import ast
import warnings
warnings.filterwarnings('ignore')
import matplotlib as mpl

mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
})

RESULTS_DIR = r"../data"
 
FOLDERS = {
    'PhemBots_NoMess':      'MainSystem_NoMess',
    'PhemBots_CMess':       'MainSystem_CMess',
    'PhemBots_Random_Mess': 'MainSystem_Random_Mess',
    'Auction_NoMess':       'Auction_NoMess',
    'Auction_CMess':        'Auction_CMess',
    'Auction_Random_Mess':  'Auction_Random_Mess',
    'Boustrophedon_Random_Mess': 'Boustrophedon_Random_Mess',
}
 
SYSTEM_COLOURS = {
    'PhemBots':         '#9C27B0',
    'Auction':      '#FF9800', 
    'Boustrophedon':'#795548',
}
 
MESS_COLOURS = {
    'red':   '#FF0000',
    'green': '#00CC00',
    'blue':  '#0000FF',
}

CONSISTENT_MESS_TIMES = [
    (30,  'red'),
    (60,  'blue'),
    (90,  'blue'),
    (90,  'green'),
    (120, 'red'),
    (150, 'green'),
]
 
GRID_ROWS = 30
GRID_COLS = 30
BORDER    = 3
TRIAL_DURATION = 180
INTERPOLATION_TIMES = np.arange(0, TRIAL_DURATION + 1, 1)
 
FIG_DPI = 300
FIG_FORMAT = 'png'
OUTPUT_DIR = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def load_trials(folder_key):
    """Load all CSV trial files from a folder, return list of DataFrames."""
    folder = os.path.join(RESULTS_DIR, FOLDERS[folder_key])
    trials = []
    if not os.path.exists(folder):
        print(f"Folder not found: {folder}")
        return trials
    for fname in sorted(os.listdir(folder)):
        if fname.endswith('.csv') and fname.startswith('trial_'):
            path = os.path.join(folder, fname)
            try:
                df = pd.read_csv(path, on_bad_lines='skip')
                df.attrs['filename'] = fname
                df.attrs['folder'] = FOLDERS[folder_key]
                trials.append(df)
            except Exception as e:
                print(f"Error loading {fname}: {e}")
    print(f"Loaded {len(trials)} trials from {FOLDERS[folder_key]}")
    return trials
 
 
def get_snapshots(df):
    """Extract snapshot rows from a trial DataFrame."""
    return df[df['type'] == 'snapshot'].copy()
 
 
def get_events(df):
    """Extract event rows from a trial DataFrame."""
    return df[df['type'] == 'event'].copy()
 
 
def get_visit_grid(df):
    filename = df.attrs.get('filename', None)
    folder = df.attrs.get('folder', None)
    if filename is None or folder is None:
        return None
    
    filepath = os.path.join(RESULTS_DIR, folder, filename)
    
    grid = []
    found_grid = False
    try:
        with open(filepath, 'r') as f:
            for line in f:
                if 'visit_counts_grid' in line:
                    found_grid = True
                    continue
                if found_grid:
                    parts = line.strip().split(',')
                    if parts[0].startswith('row_'):
                        try:
                            row_data = [float(x) for x in parts[1:GRID_COLS+1]]
                            if len(row_data) == GRID_COLS:
                                grid.append(row_data)
                        except:
                            break
    except Exception as e:
        print(f"Error reading grid from {filename}: {e}")
        return None
    
    if len(grid) == GRID_ROWS:
        return np.array(grid)
    return None
 
 
def interpolate_metric(snapshots, metric_col, times=INTERPOLATION_TIMES):
    """Interpolate a metric column onto a common time axis."""
    t = snapshots['timestamp'].astype(float).values
    v = snapshots[metric_col].astype(float).values
    return np.interp(times, t, v)
 
 
def get_mess_events(trials):
    """
    Extract mess painted and reached events from all trials.
    Returns list of dicts per trial with paint/reach times and colours.
    """
    all_mess = []
    for df in trials:
        events = get_events(df)
        painted = events[events['event'] == 'mess_painted_auto']
        reached = events[events['event'] == 'mess_reached']
        trial_messes = []
        for _, row in painted.iterrows():
            details = str(row.get('details', ''))
            colour = 'red'
            if 'colour=80' in details:
                colour = 'blue'
            elif 'colour=700' in details or 'colour=250' in details:
                colour = 'green'
            paint_time = float(row['timestamp'])
            print(f"t={paint_time:.1f} detected={colour} details={details[:100]}")
           
            reach_time = None
            
            try:
                loc = details.split('at (')[1].split(')')[0]
            except:
                loc = None
            for _, rrow in reached.iterrows():
                rdetails = str(rrow.get('details', ''))
                if float(rrow['timestamp']) > paint_time:
                    if loc and f'at ({loc})' in rdetails:
                        reach_time = float(rrow['timestamp'])
                        break
                    elif not loc:
                        reach_time = float(rrow['timestamp'])
                        break
            is_dup = any(
                m['colour'] == colour and abs(m['paint_time'] - paint_time) < 1.0
                for m in trial_messes
            )
            if not is_dup:
                trial_messes.append({
                    'paint_time': paint_time,
                    'reach_time': reach_time,
                    'colour': colour,
                })
        all_mess.append(trial_messes)
    return all_mess
 
 
def get_mean_mess_events(all_mess):
    """Average mess event times across trials by index."""
    if not all_mess:
        return []
    max_messes = max(len(m) for m in all_mess)
    mean_events = []
    for i in range(max_messes):
        paint_times = [m[i]['paint_time'] for m in all_mess if i < len(m)]
        reach_times = [m[i]['reach_time'] for m in all_mess if i < len(m) and m[i]['reach_time'] is not None]
        colours = [m[i]['colour'] for m in all_mess if i < len(m)]
        colour = max(set(colours), key=colours.count)
        mean_events.append({
            'paint_time': np.mean(paint_times),
            'reach_time': np.mean(reach_times) if reach_times else None,
            'colour': colour,
            'paint_std': np.std(paint_times),
        })
    return mean_events
 
 
def colour_segments_by_mess(ax, times, values, all_mess, base_colour, alpha=0.8):
    """
    Plot a line that changes colour when a mess is active.
    Uses mean mess events to determine active periods.
    """
    mean_mess = get_mean_mess_events(all_mess)
    if not mean_mess:
        ax.plot(times, values, color=base_colour, alpha=alpha, linewidth=2)
        return
 
    
    seg_colours = [base_colour] * len(times)
    for mess in mean_mess:
        pt = mess['paint_time']
        rt = mess['reach_time'] if mess['reach_time'] else TRIAL_DURATION
        mc = MESS_COLOURS.get(mess['colour'], base_colour)
        for j, t in enumerate(times):
            if pt <= t <= rt:
                seg_colours[j] = mc
    i = 0
    while i < len(times) - 1:
        c = seg_colours[i]
        j = i + 1
        while j < len(times) and seg_colours[j] == c:
            j += 1
        ax.plot(times[i:j], values[i:j], color=c, alpha=alpha, linewidth=2)
        i = j - 1
        i += 1
 
 
def save_fig(fig, name):
    path = os.path.join(OUTPUT_DIR, f"{name}.{FIG_FORMAT}")
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight')
    print(f"Saved: {path}")
    plt.close(fig)
 
 

In [ ]:
def compute_interior_coverage(df):
    snaps = get_snapshots(df)
    if snaps.empty:
        return None, None

    times = snaps['timestamp'].astype(float).values
    positions_raw = snaps['robot_positions'].values

    visited = set()
    coverage_over_time = []
    interior_cells = (GRID_ROWS - 2*BORDER) * (GRID_COLS - 2*BORDER)

    for pos_str in positions_raw:
        try:
            pos = ast.literal_eval(str(pos_str))
            for robot_id, (r, c) in pos.items():
                
                if BORDER <= r < GRID_ROWS - BORDER and BORDER <= c < GRID_COLS - BORDER:
                    
                    for dr in range(-1, 2):
                        for dc in range(-1, 2):
                            nr, nc = r + dr, c + dc
                            if BORDER <= nr < GRID_ROWS - BORDER and BORDER <= nc < GRID_COLS - BORDER:
                                visited.add((nr, nc))
        except:
            pass
        coverage_over_time.append(len(visited) / interior_cells * 100)

    return times, np.array(coverage_over_time)

In [ ]:
from scipy.stats import mannwhitneyu
import pandas as pd

def run_significance_tests():
    comparisons = [
        ('PhemBots No Mess',    'Auction No Mess',       'PhemBots_NoMess',        'Auction_NoMess'),
        ('PhemBots Consistent', 'Auction Consistent',    'PhemBots_CMess',          'Auction_CMess'),
        ('PhemBots Random',     'Auction Random',        'PhemBots_Random_Mess',    'Auction_Random_Mess'),
        ('PhemBots Random',     'Boustrophedon Random',  'PhemBots_Random_Mess',    'Boustrophedon_Random_Mess'),
    ]

    metrics = {
        'Boundary Visits': lambda trials: [
            sum(get_visit_grid(df)[get_visit_grid(df) is not None and True] if get_visit_grid(df) is not None else [0]
                for df in trials)
        ],
    }

    results = []

    for label_a, label_b, key_a, key_b in comparisons:
        trials_a = load_trials(key_a)
        trials_b = load_trials(key_b)

        
        bv_a, bv_b = [], []
        for df in trials_a:
            grid = get_visit_grid(df)
            if grid is not None:
                bv_a.append(grid[:BORDER,:].sum() + grid[-BORDER:,:].sum() +
                           grid[BORDER:-BORDER,:BORDER].sum() + grid[BORDER:-BORDER,-BORDER:].sum())
        for df in trials_b:
            grid = get_visit_grid(df)
            if grid is not None:
                bv_b.append(grid[:BORDER,:].sum() + grid[-BORDER:,:].sum() +
                           grid[BORDER:-BORDER,:BORDER].sum() + grid[BORDER:-BORDER,-BORDER:].sum())

       
        ird_a = [np.mean(compute_inter_robot_distance_mm(df)[1]) 
                 for df in trials_a if compute_inter_robot_distance_mm(df)[0] is not None]
        ird_b = [np.mean(compute_inter_robot_distance_mm(df)[1]) 
                 for df in trials_b if compute_inter_robot_distance_mm(df)[0] is not None]

        
        nc_a = [get_snapshots(df)['near_collisions'].astype(float).sum() for df in trials_a]
        nc_b = [get_snapshots(df)['near_collisions'].astype(float).sum() for df in trials_b]

        
        re_a, re_b = [], []
        for df in trials_a:
            snaps = get_snapshots(df)
            t, cum, _ = get_distance_traveled(df)
            if t is not None and cum[-1] > 0:
                re_a.append(float(snaps['total_reward'].dropna().iloc[-1]) / cum[-1])
        for df in trials_b:
            snaps = get_snapshots(df)
            t, cum, _ = get_distance_traveled(df)
            if t is not None and cum[-1] > 0:
                re_b.append(float(snaps['total_reward'].dropna().iloc[-1]) / cum[-1])

        for metric_name, data_a, data_b in [
            ('Boundary Visits', bv_a, bv_b),
            ('Mean IRD (mm)', ird_a, ird_b),
            ('Near Collisions', nc_a, nc_b),
            ('Reward Efficiency', re_a, re_b),
        ]:
            if data_a and data_b:
                stat, p = mannwhitneyu(data_a, data_b, alternative='two-sided')
                results.append({
                    'Metric': metric_name,
                    'Comparison': f'{label_a} vs {label_b}',
                    'U Statistic': round(stat, 1),
                    'p-value': round(p, 4),
                    'Significant': 'Yes' if p < 0.05 else 'No'
                })

    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
    return df_results

results_df = run_significance_tests()

In [ ]:
def plot_interior_coverage():
    conditions = [
        ('No Mess',         'PhemBots_NoMess',     'Auction_NoMess',     'Boustrophedon_Random_Mess'),
        ('Consistent Mess', 'PhemBots_CMess',       'Auction_CMess',      'Boustrophedon_Random_Mess'),
        ('Random Mess',     'PhemBots_Random_Mess', 'Auction_Random_Mess','Boustrophedon_Random_Mess'),
    ]

    interior_cells = (GRID_ROWS - 2*BORDER) * (GRID_COLS - 2*BORDER)

    for cond_name, main_key, auction_key, boust_key in conditions:
        fig, ax = plt.subplots(figsize=(10, 6))

        for sys_name, folder_key, colour in [
            ('PhemBots',            main_key,    SYSTEM_COLOURS['PhemBots']),
            ('Auction',                auction_key, SYSTEM_COLOURS['Auction']),
            ('Boustrophedon (Random)', boust_key,   SYSTEM_COLOURS['Boustrophedon']),
        ]:
            trials = load_trials(folder_key)
            if not trials:
                continue

            all_interp = []
            for df in trials:
                t, cov = compute_interior_coverage(df)
                if t is None:
                    continue
                all_interp.append(np.interp(INTERPOLATION_TIMES, t, cov))

            if not all_interp:
                continue

            mean_cov = np.mean(all_interp, axis=0)
            std_cov  = np.std(all_interp, axis=0)

            ax.errorbar(
                INTERPOLATION_TIMES, mean_cov,
                yerr=std_cov,
                fmt='o-', color=colour, alpha=0.8,
                markersize=3, linewidth=1.5,
                markevery=10, capsize=2,
                label=sys_name
            )

        
        ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, linewidth=1)

        if cond_name == 'Consistent Mess':
            time_counts = {}
            for t_mess, mess_colour in CONSISTENT_MESS_TIMES:
                count = time_counts.get(t_mess, 0)
                mc = MESS_COLOURS.get(mess_colour, 'gray')
                linestyle = (0, (5, 1)) if count == 0 else (0, (1, 1))
                ax.axvline(x=t_mess, color=mc, alpha=0.8, linewidth=1.5, linestyle=linestyle)
                time_counts[t_mess] = count + 1

        ax.set_xlabel('Time (s)', fontsize=18)
        ax.set_ylabel('Interior Coverage (%)', fontsize=18)
        ax.set_title(f'Interior Cell Coverage Over Time - {cond_name}\n'
                     f'(Boundary cells excluded)', fontsize=18)
        ax.legend(fontsize=18)
        ax.set_xlim(0, TRIAL_DURATION)
        ax.set_ylim(0, 105)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_fig(fig, f'interior_coverage_{cond_name.replace(" ","_").lower()}')

In [ ]:
plot_interior_coverage()

In [ ]:
def plot_coverage(include_boundary=True):
    """
    Coverage % over time for all 3 conditions.
    One graph per condition, all 3 systems on same graph.
    Generates both with and without boundary cells.
    """
    conditions = [
        ('No Mess',        'PhemBots_NoMess',    'Auction_NoMess',  'Boustrophedon_Random_Mess'),
        ('Consistent Mess','PhemBots_CMess',     'Auction_CMess',   'Boustrophedon_Random_Mess'),
        ('Random Mess',    'PhemBots_Random_Mess',    'Auction_Random_Mess',  'Boustrophedon_Random_Mess'),
    ]
 
    suffix = 'with_boundary' if include_boundary else 'no_boundary'
    valid_cells = GRID_ROWS * GRID_COLS
    if not include_boundary:
        inner = (GRID_ROWS - 2*BORDER) * (GRID_COLS - 2*BORDER)
        valid_cells = inner
 
    for cond_name, main_key, auction_key, boust_key in conditions:
        fig, ax = plt.subplots(figsize=(10, 6))
 
        for sys_name, folder_key, colour in [
            ('PhemBots', main_key, SYSTEM_COLOURS['PhemBots']),
            ('Auction',     auction_key, SYSTEM_COLOURS['Auction']),
            ('Boustrophedon (Random)', boust_key, SYSTEM_COLOURS['Boustrophedon']),
        ]:
            trials = load_trials(folder_key)
            if not trials:
                continue
 
            all_interp = []
            all_times  = []
            for df in trials:
                snaps = get_snapshots(df)
                if snaps.empty:
                    continue
                cov = snaps['coverage_pct'].astype(float).values
                if not include_boundary:
                    total = GRID_ROWS * GRID_COLS
                    boundary = total - inner
                    cov = np.clip((cov / 100 * total - boundary) / inner * 100, 0, 100)
                t = snaps['timestamp'].astype(float).values
                interp = np.interp(INTERPOLATION_TIMES, t, cov)
                all_interp.append(interp)
                all_times.append(np.interp(INTERPOLATION_TIMES, t, t))
 
            if not all_interp:
                continue
 
            mean_cov  = np.mean(all_interp, axis=0)
            std_cov   = np.std(all_interp, axis=0)
            mean_time = np.mean(all_times, axis=0)
            std_time  = np.std(all_times, axis=0)
 
            ax.errorbar(
                mean_time, mean_cov,
                xerr=std_time, yerr=std_cov,
                fmt='o-', color=colour, alpha=0.8,
                markersize=3, linewidth=1.5,
                markevery=10, capsize=2,
                label=sys_name
            )
        if include_boundary:
            interior_fraction = (GRID_ROWS - 2*BORDER) * (GRID_COLS - 2*BORDER) / (GRID_ROWS * GRID_COLS) * 100
            ax.axhline(y=interior_fraction, color='gray', linestyle='--', alpha=0.5, linewidth=1)
            ax.text(5, interior_fraction + 1, 'Max interior coverage', fontsize=18, color='gray')
        ax.set_xlabel('Time (s)', fontsize=18)
        ax.set_ylabel('Coverage (%)', fontsize=18)
        title_suffix = '' if include_boundary else ' (Boundary Excluded)'
        ax.set_title(f'Coverage Over Time - {cond_name}{title_suffix}', fontsize=20)
        ax.legend(fontsize=18)
        ax.set_xlim(0, TRIAL_DURATION)
        ax.set_ylim(0, 100)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_fig(fig, f'coverage_{cond_name.replace(" ","_").lower()}_{suffix}')

In [ ]:
plot_coverage(include_boundary=True)

In [ ]:
def plot_total_reward():
    conditions = [
        ('No Mess',         'PhemBots_NoMess',  'Auction_NoMess',  'Boustrophedon_Random_Mess'),
        ('Consistent Mess', 'PhemBots_CMess',   'Auction_CMess',   'Boustrophedon_Random_Mess'),
        ('Random Mess',     'PhemBots_Random_Mess',  'Auction_Random_Mess',  'Boustrophedon_Random_Mess'),
    ]

    line_styles = {
        'PhemBots':             '-',
        'Auction':                 '--',
        'Boustrophedon (Random)':  ':',
    }

    for cond_name, main_key, auction_key, boust_key in conditions:
        fig, ax = plt.subplots(figsize=(10, 6))
        is_random = cond_name == 'Random Mess'

        for sys_name, folder_key, colour in [
            ('PhemBots',            main_key,    SYSTEM_COLOURS['PhemBots']),
            ('Auction',                auction_key, SYSTEM_COLOURS['Auction']),
            ('Boustrophedon (Random)', boust_key,   SYSTEM_COLOURS['Boustrophedon']),
        ]:
            trials = load_trials(folder_key)
            if not trials:
                continue

            if cond_name == 'No Mess' and sys_name == 'Boustrophedon (Random)':
                all_mess = []
            else:
                all_mess = get_mess_events(trials)

            ls = line_styles[sys_name]

            if is_random:
                default_colour = '#333333'
                markers = {
                    'PhemBots':             'o',
                    'Auction':                 '^',
                    'Boustrophedon (Random)':  's',
                }
                marker = markers[sys_name]
                
                for i, df in enumerate(trials):
                    snaps = get_snapshots(df)
                    if snaps.empty or 'total_reward' not in snaps.columns:
                        continue
                    t   = snaps['timestamp'].astype(float).values
                    val = snaps['total_reward'].astype(float).values
                    interp = np.interp(INTERPOLATION_TIMES, t, val)
                    
                    trial_mess = all_mess[i] if i < len(all_mess) else []
                    
                    point_colours = []
                    for time_pt in INTERPOLATION_TIMES:
                        pt_colour = default_colour
                        for mess in trial_mess:
                            rt = mess['reach_time'] if mess['reach_time'] else TRIAL_DURATION
                            if mess['paint_time'] <= time_pt <= rt:
                                pt_colour = MESS_COLOURS.get(mess['colour'], default_colour)
                                break
                        point_colours.append(pt_colour)
                    
                    step = 5
                    ax.scatter(INTERPOLATION_TIMES[::step], interp[::step],
                              c=point_colours[::step], marker=marker,
                              s=8, alpha=0.4)

                ax.scatter([], [], c=default_colour, marker=marker, s=20, label=sys_name)

            else:
                
                all_interp = []
                for df in trials:
                    snaps = get_snapshots(df)
                    if snaps.empty or 'total_reward' not in snaps.columns:
                        continue
                    t   = snaps['timestamp'].astype(float).values
                    val = snaps['total_reward'].astype(float).values
                    all_interp.append(np.interp(INTERPOLATION_TIMES, t, val))

                if not all_interp:
                    continue

                mean_val = np.mean(all_interp, axis=0)
                std_val  = np.std(all_interp, axis=0)

                if sys_name == 'Boustrophedon (Random)':
                    ax.plot(INTERPOLATION_TIMES, mean_val, color=colour, linewidth=2, alpha=0.8)
                elif cond_name == 'Consistent Mess':
                    
                    for i, (df, trial_mess) in enumerate(zip(trials, all_mess)):
                        snaps = get_snapshots(df)
                        if snaps.empty or 'total_reward' not in snaps.columns:
                            continue
                        t = snaps['timestamp'].astype(float).values
                        val = snaps['total_reward'].astype(float).values
                        interp = np.interp(INTERPOLATION_TIMES, t, val)
                        colour_segments_by_mess(ax, INTERPOLATION_TIMES, interp,
                                               [trial_mess], colour, alpha=0.2)
                    ax.plot(INTERPOLATION_TIMES, mean_val, color=colour, linewidth=2, alpha=0.9)
                else:
                    colour_segments_by_mess(ax, INTERPOLATION_TIMES, mean_val, all_mess, colour)
                ax.fill_between(INTERPOLATION_TIMES,
                                mean_val - std_val,
                                mean_val + std_val,
                                alpha=0.15, color=colour)
                ax.plot([], [], color=colour, linestyle=ls, linewidth=2, label=sys_name)

        ax.set_xlabel('Time (s)', fontsize=18)
        ax.set_ylabel('Total Reward Collected', fontsize=18)
        ax.set_title(f'Total Reward Over Time - {cond_name}', fontsize=20)
        legend_elements = [
            Line2D([0], [0], color=SYSTEM_COLOURS['PhemBots'], linewidth=2, label='PhemBots'),
            Line2D([0], [0], color=SYSTEM_COLOURS['Auction'], linewidth=2, label='Auction'),
            Line2D([0], [0], color=SYSTEM_COLOURS['Boustrophedon'], linewidth=2, label='Boustrophedon (Random)'),
        ]
        ax.legend(handles=legend_elements, fontsize=18)
        ax.set_xlim(0, TRIAL_DURATION)
        ax.grid(True, alpha=0.3)
        if cond_name == 'Consistent Mess':
            time_counts = {}
            for t, mess_colour in CONSISTENT_MESS_TIMES:
                count = time_counts.get(t, 0)
                mc = MESS_COLOURS.get(mess_colour, 'gray')
                linestyle = '--' if count == 0 else '-'
                if count == 0:
                    ax.axvline(x=t, color=mc, alpha=0.5, linewidth=2, linestyle=(0, (5, 10)))
                else:
                    ax.axvline(x=t, color=mc, alpha=1, linewidth=1.5, linestyle=(0, (1, 1)))
                time_counts[t] = count + 1

        fig.tight_layout()
        save_fig(fig, f'total_reward_{cond_name.replace(" ","_").lower()}')

In [ ]:
plot_total_reward()

In [ ]:
def compute_inter_robot_distance_mm(df):
    """
    Calculate average inter-robot distance in mm from raw position data.
    """
    snaps = get_snapshots(df)
    if snaps.empty:
        return None, None

    cell_w = 900.0 / 30.0
    cell_h = 652.0 / 30.0

    times = snaps['timestamp'].astype(float).values
    positions_raw = snaps['robot_positions'].values
    distances = []

    for pos_str in positions_raw:
        try:
            pos = ast.literal_eval(str(pos_str))
            robot_list = list(pos.values())
            if len(robot_list) < 2:
                distances.append(0.0)
                continue
            pair_distances = []
            for i in range(len(robot_list)):
                for j in range(i + 1, len(robot_list)):
                    r1, c1 = robot_list[i]
                    r2, c2 = robot_list[j]
                    dist_mm = (((r1 - r2) * cell_h)**2 + ((c1 - c2) * cell_w)**2)**0.5
                    pair_distances.append(dist_mm)
            distances.append(np.mean(pair_distances))
        except:
            distances.append(0.0)

    return times, np.array(distances)

def plot_inter_robot_distance():
    configs = [
    ('PhemBots - No Mess',         'PhemBots_NoMess',         None),
    ('PhemBots - Consistent Mess', 'PhemBots_CMess',          'consistent'),
    ('PhemBots - Random Mess',       'PhemBots_Random_Mess',    'random'),
    ('Auction - No Mess',             'Auction_NoMess',            None),
    ('Auction - Consistent Mess',     'Auction_CMess',             'consistent'),
    ('Auction - Random Mess',         'Auction_Random_Mess',       'random'),
    ('Boustrophedon - Random Mess',   'Boustrophedon_Random_Mess', 'random'),
]

    system_colour_map = {
        'PhemBots':         SYSTEM_COLOURS['PhemBots'],
        'Auction':      SYSTEM_COLOURS['Auction'],
        'Boustrophedon':SYSTEM_COLOURS['Boustrophedon'],
    }

    for title, folder_key, mess_type in configs:
        trials = load_trials(folder_key)
        if not trials:
            continue

        sys_name = title.split(' - ')[0]
        colour = system_colour_map[sys_name.split()[0]]

        all_interp = []
        all_mess = get_mess_events(trials)
        if mess_type is None:
            all_mess = []

        for df in trials:
            snaps = get_snapshots(df)
            if snaps.empty:
                continue
            t_mm, dist_mm = compute_inter_robot_distance_mm(df)
            if t_mm is None:
                continue
            all_interp.append(np.interp(INTERPOLATION_TIMES, t_mm, dist_mm))

        if not all_interp:
            continue

        mean_val = np.mean(all_interp, axis=0)
        std_val  = np.std(all_interp, axis=0)

        fig, ax = plt.subplots(figsize=(10, 5))

        for i, (df, trial_mess) in enumerate(zip(trials, all_mess)) if all_mess else enumerate(zip(trials, [[]]*len(trials))):
            snaps = get_snapshots(df)
            if snaps.empty:
                continue
            t_mm, dist_mm = compute_inter_robot_distance_mm(df)
            if t_mm is None:
                continue
            interp = np.interp(INTERPOLATION_TIMES, t_mm, dist_mm)
            point_colours = []
            for time_pt in INTERPOLATION_TIMES:
                pt_colour = colour
                if mess_type == 'consistent':
                    for t_mess, mess_colour in CONSISTENT_MESS_TIMES:
                        mc = MESS_COLOURS.get(mess_colour, colour)
                        reach_t = TRIAL_DURATION
                        for m in trial_mess:
                            if abs(m['paint_time'] - t_mess) < 2.0 and m['colour'] == mess_colour:
                                reach_t = m['reach_time'] if m['reach_time'] else TRIAL_DURATION
                                break
                        if t_mess <= time_pt <= reach_t:
                            pt_colour = mc
                            break
                else:
                    for mess in (trial_mess if trial_mess else []):
                        paint_t = mess['paint_time']
                        reach_t = mess['reach_time'] if mess['reach_time'] else TRIAL_DURATION
                        if paint_t <= time_pt <= reach_t:
                            pt_colour = MESS_COLOURS.get(mess['colour'], colour)
                            break
                point_colours.append(pt_colour)
            ax.scatter(INTERPOLATION_TIMES, interp, c=point_colours, s=2, alpha=0.3, zorder=2)

        if mess_type == 'consistent':
            ax.plot(INTERPOLATION_TIMES, mean_val, color='black', linewidth=2.5, alpha=0.9, zorder=5)
            for t_mess, mess_colour in CONSISTENT_MESS_TIMES:
                mc = MESS_COLOURS.get(mess_colour, 'gray')
                ax.axvline(x=t_mess, color=mc, alpha=0.6, linewidth=1.5, linestyle='--')
        elif mess_type == 'random':
            ax.plot(INTERPOLATION_TIMES, mean_val, color=colour, linewidth=2.5, alpha=0.9)
        else:
            ax.plot(INTERPOLATION_TIMES, mean_val, color=colour, linewidth=2.5, alpha=0.9)
        step = 10
        ax.errorbar(
            INTERPOLATION_TIMES[::step], mean_val[::step],
            yerr=std_val[::step],
            fmt='none', color=colour, alpha=0.5, capsize=3, linewidth=1
        )

        ax.set_xlabel('Time (s)', fontsize=18)
        ax.set_ylabel('Average Inter-Robot Distance (mm)', fontsize=18)
        ax.set_title(title, fontsize=20)
        ax.set_xlim(0, TRIAL_DURATION)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_fig(fig, f'inter_robot_{folder_key.lower()}')

In [ ]:
plot_inter_robot_distance()

In [ ]:
def plot_inter_robot_distance_boxplot():
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',           'Auction_NoMess'),
        ('Auction\nConsistent',        'Auction_CMess'),
        ('Auction\nRandom',            'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',      'Boustrophedon_Random_Mess'),
    ]

    box_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]

    all_data = []
    labels = []

    for label, folder_key in configs:
        trials = load_trials(folder_key)
        trial_means = []
        for df in trials:
            t_mm, dist_mm = compute_inter_robot_distance_mm(df)
            if t_mm is None:
                continue
            trial_means.append(np.mean(dist_mm))
        all_data.append(trial_means)
        labels.append(label)

    fig, ax = plt.subplots(figsize=(12, 6))
    bp = ax.boxplot(all_data, patch_artist=True, notch=False,
                    showfliers=False,
                    medianprops=dict(color='black', linewidth=2))

    for patch, colour in zip(bp['boxes'], box_colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)

    for whisker in bp['whiskers']:
        whisker.set_color('black')
    for cap in bp['caps']:
        cap.set_color('black')

    for i, (data, colour) in enumerate(zip(all_data, box_colours)):
        x = np.random.normal(i + 1, 0.05, size=len(data))
        ax.scatter(x, data, alpha=0.6, color=colour,
                  edgecolor='black', linewidth=0.5, s=30, zorder=3)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Mean Inter-Robot Distance (mm)', fontsize=18)
    ax.set_title('Mean Inter-Robot Distance per Trial', fontsize=20)
    ax.grid(True, alpha=0.3, axis='y')

    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], alpha=0.7, label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], alpha=0.7, label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], alpha=0.7, label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'inter_robot_distance_boxplot')

In [ ]:
plot_inter_robot_distance_boxplot()

In [ ]:
trials = load_trials('PhemBots_NoMess')
for i, df in enumerate(trials):
    t_mm, dist_mm = compute_inter_robot_distance_mm(df)
    if t_mm is not None:
        print(f"Trial {i+1}: mean={np.mean(dist_mm):.0f}mm std={np.std(dist_mm):.0f}mm")

In [ ]:
def plot_near_collisions_step():
    configs = [
        ('PhemBots - No Mess',         'PhemBots_NoMess',         None),
        ('PhemBots - Consistent Mess', 'PhemBots_CMess',          'consistent'),
        ('PhemBots - Random Mess',       'PhemBots_Random_Mess',    'random'),
        ('Auction - No Mess',             'Auction_NoMess',            None),
        ('Auction - Consistent Mess',     'Auction_CMess',             'consistent'),
        ('Auction - Random Mess',         'Auction_Random_Mess',       'random'),
        ('Boustrophedon - Random Mess',   'Boustrophedon_Random_Mess', 'random'),
    ]

    system_colour_map = {
        'PhemBots':         SYSTEM_COLOURS['PhemBots'],
        'Auction':      SYSTEM_COLOURS['Auction'],
        'Boustrophedon':SYSTEM_COLOURS['Boustrophedon'],
    }

    for title, folder_key, mess_type in configs:
        trials = load_trials(folder_key)
        if not trials:
            continue

        sys_name = title.split(' - ')[0]
        colour = system_colour_map[sys_name.split()[0]]

        all_interp = []
        all_mess = get_mess_events(trials) if mess_type else []

        for df in trials:
            snaps = get_snapshots(df)
            if snaps.empty:
                continue
            t   = snaps['timestamp'].astype(float).values
            val = snaps['near_collisions'].astype(float).values
            all_interp.append(np.interp(INTERPOLATION_TIMES, t, val))

        if not all_interp:
            continue

        mean_val = np.mean(all_interp, axis=0)

        fig, ax = plt.subplots(figsize=(10, 4))

        for interp in all_interp:
            ax.step(INTERPOLATION_TIMES, interp, where='post',
                   color=colour, linewidth=0.8, alpha=0.2)

        ax.step(INTERPOLATION_TIMES, mean_val, where='post',
               color='black', linewidth=2.5, alpha=0.9, zorder=5)
        if mess_type == 'consistent':
            time_counts = {}
            for t_mess, mess_colour in CONSISTENT_MESS_TIMES:
                count = time_counts.get(t_mess, 0)
                mc = MESS_COLOURS.get(mess_colour, 'gray')
                linestyle = (0, (5, 1)) if count == 0 else (0, (1, 1))
                ax.axvline(x=t_mess, color=mc, alpha=0.8, linewidth=2, linestyle=linestyle)
                time_counts[t_mess] = count + 1

        ax.set_xlabel('Time (s)', fontsize=18)
        ax.set_ylabel('Near Collision Count', fontsize=18)
        ax.set_title(title, fontsize=20)
        ax.set_xlim(0, TRIAL_DURATION)
        ax.set_ylim(-0.1, 3.5)
        ax.set_yticks([0, 1, 2, 3])
        ax.grid(True, alpha=0.3)

        overall_mean = np.mean(mean_val)
        ax.text(0.98, 0.95, f'Trial mean: {overall_mean:.3f}',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=18, style='italic',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

        if mess_type == 'consistent':
            legend_elements = [
                Line2D([0], [0], color='red', linewidth=1.5, linestyle='--', label='Red\nmess'),
                Line2D([0], [0], color='green', linewidth=1.5, linestyle='--', label='Green\nmess'),
                Line2D([0], [0], color='blue', linewidth=1.5, linestyle='--', label='Blue\nmess'),
            ]
            ax.legend(handles=legend_elements, fontsize=15, loc='upper left')

        fig.tight_layout()
        save_fig(fig, f'near_collisions_{folder_key.lower()}')

In [ ]:
plot_near_collisions_step()

In [ ]:
def plot_near_collision_summary():
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',       'Auction_NoMess'),
        ('Auction\nConsistent',    'Auction_CMess'),
        ('Auction\nRandom',        'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',  'Boustrophedon_Random_Mess'),
    ]

    bar_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]

    means, stds, labels = [], [], []

    for label, folder_key in configs:
        trials = load_trials(folder_key)
        trial_means = []
        for df in trials:
            snaps = get_snapshots(df)
            if snaps.empty:
                continue
            val = snaps['near_collisions'].astype(float).values
            trial_means.append(np.mean(val))
        if trial_means:
            means.append(np.mean(trial_means))
            stds.append(np.std(trial_means))
        else:
            means.append(0)
            stds.append(0)
        labels.append(label)

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=stds, capsize=5, color=bar_colours,
           alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Mean Near Collision Count', fontsize=18)
    ax.set_title('Mean Near Collision Count Across Trial\n'
                 '(PhemBots collisions during mess response are intentional)', fontsize=18)
    ax.grid(True, alpha=0.3, axis='y')

    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'near_collision_summary_bar')

In [ ]:
plot_near_collision_summary()

In [ ]:
def plot_near_collision_boxplot():
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',       'Auction_NoMess'),
        ('Auction\nConsistent',    'Auction_CMess'),
        ('Auction\nRandom',        'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',  'Boustrophedon_Random_Mess'),
    ]

    box_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]

    all_data = []
    labels = []

    for label, folder_key in configs:
        trials = load_trials(folder_key)
        trial_totals = []
        for df in trials:
            snaps = get_snapshots(df)
            if snaps.empty:
                continue
            val = snaps['near_collisions'].astype(float).values

            trial_totals.append(np.sum(val > 0))
        all_data.append(trial_totals)
        labels.append(label)

    fig, ax = plt.subplots(figsize=(12, 6))
    bp = ax.boxplot(all_data, patch_artist=True, notch=False,
                    showfliers=False,
                    medianprops=dict(color='black', linewidth=2))

    for patch, colour in zip(bp['boxes'], box_colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)

    for whisker in bp['whiskers']:
        whisker.set_color('black')
    for cap in bp['caps']:
        cap.set_color('black')
    for flier in bp['fliers']:
        flier.set(marker='o', markerfacecolor='black', markersize=4, alpha=0.5)
    for i, data in enumerate(all_data):
        x = np.random.normal(i + 1, 0.05, size=len(data))
        ax.scatter(x, data, alpha=0.6, color=box_colours[i], edgecolor='black',
                  linewidth=0.5, s=30, zorder=3)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Total Snapshots with Near Collision', fontsize=18)
    ax.set_title('Near Collision Frequency per Trial\n'
                 '(PhemBots collisions during mess response are intentional)', fontsize=18)
    ax.grid(True, alpha=0.3, axis='y')

    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], alpha=0.7, label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], alpha=0.7, label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], alpha=0.7, label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'near_collision_boxplot')

In [ ]:
plot_near_collision_boxplot()

In [ ]:
ARENA_WIDTH_M  = 0.9
ARENA_HEIGHT_M = 0.652

def plot_visit_heatmaps():
    """
    Averaged cell visit frequency heatmap for random mess condition only.
    Consistent colour scale across all heatmaps.
    """
    configs = [
        ('PhemBots - Random Mess',       'PhemBots_Random_Mess'),
        ('Auction - Random Mess',         'Auction_Random_Mess'),
        ('Boustrophedon - Random Mess',   'Boustrophedon_Random_Mess'),
    ]
    all_grids = {}
    global_max = 0
    for title, folder_key in configs:
        trials = load_trials(folder_key)
        grids = [get_visit_grid(df) for df in trials]
        grids = [g for g in grids if g is not None]
        if grids:
            mean_grid = np.mean(grids, axis=0)
            all_grids[folder_key] = (title, mean_grid)
            global_max = max(global_max, mean_grid.max())

    titles_and_grids = [(title, mean_grid) for folder_key, (title, mean_grid) in all_grids.items()]

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))

    positions = [axes[0,0], axes[0,1], axes[1,0]]

    for ax, (title, mean_grid) in zip(positions, titles_and_grids):
        im = ax.imshow(mean_grid, cmap='viridis', vmin=0, vmax=global_max,
                       aspect=ARENA_HEIGHT_M/ARENA_WIDTH_M, origin='upper')
        border_rect = plt.Rectangle((BORDER-0.5, BORDER-0.5),
                                     GRID_COLS - 2*BORDER,
                                     GRID_ROWS - 2*BORDER,
                                     linewidth=2, edgecolor='cyan',
                                     facecolor='none', linestyle='--')
        ax.add_patch(border_rect)
        ax.set_title(title, fontsize=18)
        ax.set_xticks([])
        ax.set_yticks([])

    axes[1,1].set_axis_off()
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    cbar_ax = inset_axes(axes[1,1], width='80%', height='30%', loc='center')
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Mean Visit Count', fontsize=18)

    fig.suptitle('Cell Visit Frequency Heatmaps - Random Mess Condition', fontsize=20)
    fig.tight_layout()
    save_fig(fig, 'heatmaps_combined')

In [ ]:
plot_visit_heatmaps()

In [ ]:
def plot_reward_per_cell():
    """Bar chart of reward per cell visited for each system/condition."""
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',       'Auction_NoMess'),
        ('Auction\nConsistent',    'Auction_CMess'),
        ('Auction\nRandom',        'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',  'Boustrophedon_Random_Mess'),
    ]
 
    bar_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]
 
    means, stds, labels = [], [], []
 
    for label, folder_key in configs:
        trials = load_trials(folder_key)
        values = []
        for df in trials:
            summary = df[df['event'] == 'trial_end']
            snaps = get_snapshots(df)
            if summary.empty or snaps.empty:
                continue
            try:
                total_reward = float(snaps['total_reward'].dropna().iloc[-1])
                cells = snaps['coverage_pct'].dropna().iloc[-1] / 100 * GRID_ROWS * GRID_COLS
                if cells > 0:
                    values.append(total_reward / cells)
            except:
                continue
        if values:
            means.append(np.mean(values))
            stds.append(np.std(values))
        else:
            means.append(0)
            stds.append(0)
        labels.append(label)
 
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(labels))
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=bar_colours,
                  alpha=0.8, edgecolor='black', linewidth=0.5)
 
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Reward per Cell Visited', fontsize=18)
    ax.set_title('Reward Efficiency: Total Reward per Cell Visited', fontsize=20)
    ax.grid(True, alpha=0.3, axis='y')
 
    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)

    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 50,
                f'{mean:.0f}', ha='center', va='bottom', fontsize=18)
 
    fig.tight_layout()
    save_fig(fig, 'reward_per_cell_bar')

In [ ]:
plot_reward_per_cell()

In [ ]:
def plot_reward_per_distance():
    configs = [
        ('No Mess',          'PhemBots_NoMess'),
        ('Consistent',       'PhemBots_CMess'),
        ('Random',           'PhemBots_Random_Mess'),
        ('No Mess',       'Auction_NoMess'),
        ('Consistent',    'Auction_CMess'),
        ('Random',        'Auction_Random_Mess'),
        ('Random',  'Boustrophedon_Random_Mess'),
    ]

    bar_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]

    means, stds, labels = [], [], []

    for label, folder_key in configs:
        trials = load_trials(folder_key)
        values = []
        for df in trials:
            snaps = get_snapshots(df)
            if snaps.empty or 'total_reward' not in snaps.columns:
                continue
            try:
                total_reward = float(snaps['total_reward'].dropna().iloc[-1])
                t, cum, spd = get_distance_traveled(df)
                if cum is None or cum[-1] == 0:
                    continue
                total_distance = cum[-1]
                values.append(total_reward / total_distance)
            except:
                continue
        if values:
            means.append(np.mean(values))
            stds.append(np.std(values))
        else:
            means.append(0)
            stds.append(0)
        labels.append(label)

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(labels))
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=bar_colours,
                  alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Reward per mm Traveled', fontsize=18)
    ax.set_title('Reward Efficiency: Total Reward per Distance Traveled', fontsize=20)
    ax.grid(True, alpha=0.3, axis='y')

    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.5,
                f'{mean:.0f}', ha='center', va='bottom', fontsize=13)

    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'reward_per_distance_bar')

In [ ]:
plot_reward_per_distance()

In [ ]:
def plot_mess_response_time():
    """
    Mess response time and efficiency for mess conditions only.
    Grouped by mess colour (red/green/blue) and system.
    """
    mess_conditions = [
        ('PhemBots_CMess',     'PhemBots\nConsistent', SYSTEM_COLOURS['PhemBots']),
        ('PhemBots_Random_Mess',    'PhemBots\nRandom',     SYSTEM_COLOURS['PhemBots']),
        ('Auction_CMess',  'Auction\nConsistent',     SYSTEM_COLOURS['Auction']),
        ('Auction_Random_Mess', 'Auction\nRandom',         SYSTEM_COLOURS['Auction']),
        ('Boustrophedon_Random_Mess',  'Boustrophedon\nRandom',   SYSTEM_COLOURS['Boustrophedon']),
    ]
 
    for metric, ylabel, title_suffix, fig_name in [
        ('response_time', 'Response Time (s)', 'Mess Response Time', 'mess_response_time'),
        ('efficiency',    'Efficiency (cells/s)', 'Mess Response Efficiency\n(Distance / Response Time)', 'mess_efficiency'),
    ]:
        fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=False)
        colours_list = ['red', 'green', 'blue']
 
        for ax, mess_colour in zip(axes, colours_list):
            group_labels = []
            group_means  = []
            group_stds   = []
            group_colours = []
 
            for folder_key, label, colour in mess_conditions:
                trials = load_trials(folder_key)
                values = []
                for df in trials:
                    events = get_events(df)
                    reached = events[events['event'] == 'mess_reached']
                    painted = events[events['event'] == 'mess_painted_auto']
 
                    for _, rrow in reached.iterrows():
                        details = str(rrow.get('details', ''))
                        rt = float(rrow['timestamp'])
                        paint_match = painted[painted['timestamp'].astype(float) < rt]
                        if paint_match.empty:
                            continue
                        paint_details = str(paint_match.iloc[-1].get('details', ''))
 
                        if 'colour=1500' in details:
                            actual_colour = 'red'
                        elif 'colour=700' in details:
                            actual_colour = 'green'
                        elif 'colour=80' in details:
                            actual_colour = 'blue'
                        else:
                            continue

                        if actual_colour != mess_colour:
                            continue
 
                        try:
                            if metric == 'response_time':
                                val = float(details.split('response_time=')[1].split('s')[0])
                            else:
                                val = float(details.split('efficiency=')[1].split()[0])
                            if metric == 'response_time' and val < 0.5:
                                continue
                            if metric == 'efficiency' and val > 20:
                                continue
                            values.append(val)
                        except:
                            continue
 
                if values:
                    group_labels.append(label)
                    group_means.append(np.mean(values))
                    group_stds.append(np.std(values))
                    group_colours.append(colour)
 
            if group_means:
                x = np.arange(len(group_labels))
                bars = ax.bar(x, group_means, yerr=group_stds, capsize=5,
                              color=group_colours, alpha=0.8,
                              edgecolor='black', linewidth=0.5)
                ax.set_xticks(x)
                ax.set_xticklabels(group_labels, fontsize=18)
                ax.set_title(f'{mess_colour.capitalize()} Mess', fontsize=18,
                             color=MESS_COLOURS[mess_colour])
                ax.grid(True, alpha=0.3, axis='y')
 
        all_max = max(ax.get_ylim()[1] for ax in axes)
        for ax in axes:
            ax.set_ylim(0, all_max)

        fig.suptitle(f'{title_suffix}', fontsize=20)
        axes[0].set_ylabel(ylabel, fontsize=18)
        fig.tight_layout()
        save_fig(fig, fig_name)

In [ ]:
plot_mess_response_time()

In [ ]:
def plot_mess_response_boxplot():
    mess_conditions = [
        ('PhemBots_CMess',     'PhemBots\nConsistent', SYSTEM_COLOURS['PhemBots']),
        ('PhemBots_Random_Mess','PhemBots\nRandom',    SYSTEM_COLOURS['PhemBots']),
        ('Auction_CMess',        'Auction\nConsistent',     SYSTEM_COLOURS['Auction']),
        ('Auction_Random_Mess',  'Auction\nRandom',         SYSTEM_COLOURS['Auction']),
        ('Boustrophedon_Random_Mess', 'Boustrophedon\nRandom', SYSTEM_COLOURS['Boustrophedon']),
    ]

    for metric, ylabel, title_suffix, fig_name in [
        ('response_time', 'Response Time (s)',    'Mess Response Time',       'mess_response_time_boxplot'),
        ('efficiency',    'Efficiency (cells/s)', 'Mess Response Efficiency', 'mess_efficiency_boxplot'),
    ]:
        fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
        colours_list = ['red', 'green', 'blue']

        global_max = 0

        for ax, mess_colour in zip(axes, colours_list):
            all_data = []
            labels = []
            box_colours = []

            for folder_key, label, colour in mess_conditions:
                trials = load_trials(folder_key)
                values = []
                for df in trials:
                    events = get_events(df)
                    reached = events[events['event'] == 'mess_reached']
                    painted = events[events['event'] == 'mess_painted_auto']

                    for _, rrow in reached.iterrows():
                        details = str(rrow.get('details', ''))
                        rt = float(rrow['timestamp'])
                        paint_match = painted[painted['timestamp'].astype(float) < rt]
                        if paint_match.empty:
                            continue
                        paint_details = str(paint_match.iloc[-1].get('details', ''))

                        if 'colour=1500' in details:
                            actual_colour = 'red'
                        elif 'colour=700' in details:
                            actual_colour = 'green'
                        elif 'colour=80' in details:
                            actual_colour = 'blue'
                        else:
                            continue

                        if actual_colour != mess_colour:
                            continue

                        try:
                            if metric == 'response_time':
                                val = float(details.split('response_time=')[1].split('s')[0])
                            else:
                                val = float(details.split('efficiency=')[1].split()[0])
                            if metric == 'response_time' and val < 0.5:
                                continue
                            if metric == 'efficiency' and val > 20:
                                continue
                            values.append(val)
                        except:
                            continue

                all_data.append(values)
                labels.append(label)
                box_colours.append(colour)

            if not any(all_data):
                continue

            bp = ax.boxplot(all_data, patch_artist=True, notch=False,
                           showfliers=False,
                           medianprops=dict(color='black', linewidth=2))

            for patch, colour in zip(bp['boxes'], box_colours):
                patch.set_facecolor(colour)
                patch.set_alpha(0.7)

            for whisker in bp['whiskers']:
                whisker.set_color('black')
            for cap in bp['caps']:
                cap.set_color('black')

            for i, (data, colour) in enumerate(zip(all_data, box_colours)):
                x = np.random.normal(i + 1, 0.05, size=len(data))
                ax.scatter(x, data, alpha=0.5, color=colour,
                          edgecolor='black', linewidth=0.5, s=20, zorder=3)

            ax.set_xticks(range(1, len(labels) + 1))
            ax.set_xticklabels(labels, fontsize=18)
            ax.set_title(f'{mess_colour.capitalize()} Mess', fontsize=18,
                        color=MESS_COLOURS[mess_colour])
            ax.grid(True, alpha=0.3, axis='y')

            current_max = ax.get_ylim()[1]
            global_max = max(global_max, current_max)

        for ax in axes:
            ax.set_ylim(0, global_max)

        fig.suptitle(title_suffix, fontsize=20)
        axes[0].set_ylabel(ylabel, fontsize=18)
        fig.tight_layout()
        save_fig(fig, fig_name)

In [ ]:
plot_mess_response_boxplot()

In [ ]:
def plot_mess_response_boxplot_mm():
    mess_conditions = [
        ('PhemBots_CMess',          'PB\nCON', SYSTEM_COLOURS['PhemBots']),
        ('PhemBots_Random_Mess',    'PB\nRAN',     SYSTEM_COLOURS['PhemBots']),
        ('Auction_CMess',             'AUC\nCON',     SYSTEM_COLOURS['Auction']),
        ('Auction_Random_Mess',       'AUC\nRAN',         SYSTEM_COLOURS['Auction']),
        ('Boustrophedon_Random_Mess', 'BOU\nRAN',   SYSTEM_COLOURS['Boustrophedon']),
    ]

    cell_w = 900.0 / 30.0
    cell_h = 652.0 / 30.0

    for metric, ylabel, title_suffix, fig_name in [
        ('response_time', 'Response Time (s)',    'Mess Response Time',                    'mess_response_time_boxplot_mm'),
        ('efficiency',    'Efficiency (mm/s)',    'Mess Response Efficiency (mm/s)',        'mess_efficiency_boxplot_mm'),
    ]:
        fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
        colours_list = ['red', 'green', 'blue']
        global_max = 0

        for ax, mess_colour in zip(axes, colours_list):
            all_data = []
            labels = []
            box_colours = []

            for folder_key, label, colour in mess_conditions:
                trials = load_trials(folder_key)
                values = []
                for df in trials:
                    events = get_events(df)
                    reached = events[events['event'] == 'mess_reached']
                    painted = events[events['event'] == 'mess_painted_auto']

                    for _, rrow in reached.iterrows():
                        details = str(rrow.get('details', ''))
                        rt = float(rrow['timestamp'])
                        paint_match = painted[painted['timestamp'].astype(float) < rt]
                        if paint_match.empty:
                            continue
                        paint_details = str(paint_match.iloc[-1].get('details', ''))

                        if 'colour=1500' in details:
                            actual_colour = 'red'
                        elif 'colour=700' in details:
                            actual_colour = 'green'
                        elif 'colour=80' in details:
                            actual_colour = 'blue'
                        else:
                            if 'colour=1500' in paint_details:
                                actual_colour = 'red'
                            elif 'colour=700' in paint_details:
                                actual_colour = 'green'
                            elif 'colour=80' in paint_details:
                                actual_colour = 'blue'
                            else:
                                continue

                        if actual_colour != mess_colour:
                            continue

                        try:
                            response_time = float(details.split('response_time=')[1].split('s')[0])
                            if response_time < 1:
                                continue

                            if metric == 'response_time':
                                values.append(response_time)
                            else:
                                mess_loc_str = None
                                try:
                                    mess_loc_str = details.split('at (')[1].split(')')[0]
                                except:
                                    pass

                                paint_row = paint_match.iloc[-1]
                                paint_det = str(paint_row.get('details', ''))

                                if mess_loc_str is None:
                                    try:
                                        mess_loc_str = paint_det.split('at (')[1].split(')')[0]
                                    except:
                                        continue

                                try:
                                    mess_r, mess_c = map(int, mess_loc_str.split(','))
                                    robot_pos = ast.literal_eval(str(paint_row['robot_positions']))
                                except:
                                    continue

                                min_dist_mm = float('inf')
                                for rid, (r, c) in robot_pos.items():
                                    dist_mm = (((r - mess_r) * cell_h)**2 + ((c - mess_c) * cell_w)**2)**0.5
                                    min_dist_mm = min(min_dist_mm, dist_mm)

                                if min_dist_mm == float('inf') or response_time == 0:
                                    continue

                                val = min_dist_mm / response_time
                                values.append(val)
                        except:
                            continue

                all_data.append(values)
                labels.append(label)
                box_colours.append(colour)

            if not any(all_data):
                continue

            bp = ax.boxplot(all_data, patch_artist=True, notch=False,
                           showfliers=False,
                           medianprops=dict(color='black', linewidth=2))

            for patch, colour in zip(bp['boxes'], box_colours):
                patch.set_facecolor(colour)
                patch.set_alpha(0.7)

            for whisker in bp['whiskers']:
                whisker.set_color('black')
            for cap in bp['caps']:
                cap.set_color('black')

            for i, (data, colour) in enumerate(zip(all_data, box_colours)):
                x = np.random.normal(i + 1, 0.05, size=len(data))
                ax.scatter(x, data, alpha=0.5, color=colour,
                          edgecolor='black', linewidth=0.5, s=20, zorder=3)

            ax.set_xticks(range(1, len(labels) + 1))
            ax.set_xticklabels(labels, fontsize=18)
            ax.set_title(f'{mess_colour.capitalize()} Mess', fontsize=18,
                        color=MESS_COLOURS[mess_colour])
            ax.grid(True, alpha=0.3, axis='y')

            current_max = ax.get_ylim()[1]
            global_max = max(global_max, current_max)

        for ax in axes:
            ax.set_ylim(0, global_max)

        fig.suptitle(title_suffix, fontsize=20)
        axes[0].set_ylabel(ylabel, fontsize=18)
        fig.tight_layout()
        save_fig(fig, fig_name)

In [ ]:
plot_mess_response_boxplot_mm()

In [ ]:
trials = load_trials('Auction_CMess')
for i, df in enumerate(trials):
    events = get_events(df)
    reached = events[events['event'] == 'mess_reached']
    painted = events[events['event'] == 'mess_painted_auto']
    cell_w = 900.0/30.0
    cell_h = 652.0/30.0
    for _, rrow in reached.iterrows():
        details = str(rrow.get('details', ''))
        try:
            rt_val = float(details.split('response_time=')[1].split('s')[0])
            if rt_val < 1.0:
                continue
            paint_match = painted[painted['timestamp'].astype(float) < float(rrow['timestamp'])]
            if paint_match.empty:
                continue
            paint_det = str(paint_match.iloc[-1].get('details', ''))
            if 'colour=80' not in paint_det:
                continue
            mess_loc = paint_det.split('at (')[1].split(')')[0]
            mess_r, mess_c = map(int, mess_loc.split(','))
            robot_pos = ast.literal_eval(str(paint_match.iloc[-1]['robot_positions']))
            min_dist = min((((r-mess_r)*cell_h)**2+((c-mess_c)*cell_w)**2)**0.5 
                          for rid,(r,c) in robot_pos.items())
            eff = min_dist / rt_val
            if eff > 200:
                print(f"Trial {i+1} t={float(rrow['timestamp']):.1f} rt={rt_val:.2f}s dist={min_dist:.0f}mm eff={eff:.0f}")
        except:
            continue

In [ ]:
def plot_mess_response_gantt():
    systems = [
        ('PhemBots_CMess',  'PhemBots',  SYSTEM_COLOURS['PhemBots']),
        ('Auction_CMess',     'Auction',   SYSTEM_COLOURS['Auction']),
    ]
    expected_times = [30, 60, 90, 90, 120, 150]

    fig, axes = plt.subplots(2, 1, figsize=(14, 14), sharex=True)

    for ax, (folder_key, sys_name, colour) in zip(axes, systems):
        trials = load_trials(folder_key)
        if not trials:
            continue

        yticks = []
        yticklabels = []
        mess_reach_times = {}

        for trial_idx, df in enumerate(trials):
            y = trial_idx
            yticks.append(y)
            yticklabels.append(f'Trial {trial_idx + 1}')

            events = get_events(df)
            painted = events[events['event'] == 'mess_painted_auto']
            reached = events[events['event'] == 'mess_reached']

            trial_mess_bars = []
            for _, prow in painted.iterrows():
                pdetails = str(prow.get('details', ''))
                paint_time = float(prow['timestamp'])

                if 'colour=1500' in pdetails:
                    mc = 'red'
                elif 'colour=700' in pdetails:
                    mc = 'green'
                elif 'colour=80' in pdetails:
                    mc = 'blue'
                else:
                    continue

                try:
                    loc = pdetails.split('at (')[1].split(')')[0]
                except:
                    loc = None

                reach_time = None
                for _, rrow in reached.iterrows():
                    rdetails = str(rrow.get('details', ''))
                    rt = float(rrow['timestamp'])
                    if rt > paint_time:
                        if loc and f'at ({loc})' in rdetails:
                            reach_time = rt
                            break
                        elif not loc:
                            reach_time = rt
                            break

                if reach_time is None:
                    reach_time = TRIAL_DURATION

                trial_mess_bars.append({
                    'paint_time': paint_time,
                    'reach_time': reach_time,
                    'colour': mc
                })
            deduplicated = []
            for bar in trial_mess_bars:
                is_dup = False
                for existing in deduplicated:
                    if (bar['colour'] == existing['colour'] and
                        abs(bar['paint_time'] - existing['paint_time']) < 1.0):
                        is_dup = True
                        break
                if not is_dup:
                    deduplicated.append(bar)
            trial_mess_bars = deduplicated

            for bar in trial_mess_bars:
                scheduled = min(expected_times, key=lambda t: abs(t - bar['paint_time']))
                key = (scheduled, bar['colour'], round(bar['paint_time'] / 30) * 30)
                if key not in mess_reach_times:
                    mess_reach_times[key] = []
                mess_reach_times[key].append(bar['reach_time'])

            trial_mess_bars.sort(key=lambda x: x['paint_time'])

            bar_height = 0.15
            slots = []

            for bar in trial_mess_bars:
                slot = None
                for s, end_time in enumerate(slots):
                    if bar['paint_time'] >= end_time:
                        slot = s
                        slots[s] = bar['reach_time']
                        break
                if slot is None:
                    slot = len(slots)
                    slots.append(bar['reach_time'])

                n_slots = len(slots)
                bar_spacing = 0.4
                offset = (slot - (n_slots - 1) / 2.0) * bar_spacing
                bar_colour = MESS_COLOURS.get(bar['colour'], 'gray')
                duration = bar['reach_time'] - bar['paint_time']

                ax.barh(y + offset, duration, left=bar['paint_time'],
                       height=bar_height, color=bar_colour,
                       alpha=0.7, edgecolor='black', linewidth=0.5)

        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels, fontsize=18)
        ax.set_xlim(0, TRIAL_DURATION)
        ax.set_title(sys_name, fontsize=20)
        ax.grid(True, alpha=0.3, axis='x')

        for key, reach_times in mess_reach_times.items():
            if reach_times:
                mean_rt = np.mean(reach_times)
                mc = key[1]
                bar_colour = MESS_COLOURS.get(mc, 'gray')
                ax.axvline(x=mean_rt, color=bar_colour, linewidth=2,
                          linestyle=':', alpha=0.9)
                ax.text(mean_rt + 0.5, -0.5, f'μ={mean_rt:.1f}s',
                       color=bar_colour, fontsize=0, va='top')

        time_counts = {}
        for t_mess, mess_colour in CONSISTENT_MESS_TIMES:
            count = time_counts.get(t_mess, 0)
            mc = MESS_COLOURS.get(mess_colour, 'gray')
            linestyle = (0, (5, 1)) if count == 0 else (0, (1, 1))
            ax.axvline(x=t_mess, color=mc, alpha=0.5, linewidth=1.5, linestyle=linestyle)
            time_counts[t_mess] = count + 1

        legend_elements = [
            mpatches.Patch(color=MESS_COLOURS['red'], alpha=0.7, label='Red\nmess'),
            mpatches.Patch(color=MESS_COLOURS['green'], alpha=0.7, label='Green\nmess'),
            mpatches.Patch(color=MESS_COLOURS['blue'], alpha=0.7, label='Blue\nmess'),
        ]
        ax.legend(handles=legend_elements, fontsize=18, loc='upper left')

    axes[-1].set_xlabel('Time (s)', fontsize=18)
    fig.suptitle('Mess Response Duration - Consistent Mess\n(Bar length = time to clean mess)', fontsize=20)
    fig.tight_layout()
    save_fig(fig, 'mess_gantt_combined')

In [ ]:
plot_mess_response_gantt()

In [ ]:
def plot_boundary_visits():
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',       'Auction_NoMess'),
        ('Auction\nConsistent',    'Auction_CMess'),
        ('Auction\nRandom',        'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',  'Boustrophedon_Random_Mess'),
    ]

    box_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]

    all_data = []
    labels = []

    for label, folder_key in configs:
        trials = load_trials(folder_key)
        trial_values = []
        for df in trials:
            grid = get_visit_grid(df)
            if grid is None:
                continue
            border_visits = (
                grid[:BORDER, :].sum() +
                grid[-BORDER:, :].sum() +
                grid[BORDER:-BORDER, :BORDER].sum() +
                grid[BORDER:-BORDER, -BORDER:].sum()
            )
            trial_values.append(border_visits)
        all_data.append(trial_values)
        labels.append(label)

    fig, ax = plt.subplots(figsize=(12, 6))
    bp = ax.boxplot(all_data, patch_artist=True, notch=False,
                    showfliers=False,
                    medianprops=dict(color='black', linewidth=2))

    for patch, colour in zip(bp['boxes'], box_colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)

    for whisker in bp['whiskers']:
        whisker.set_color('black')
    for cap in bp['caps']:
        cap.set_color('black')

    for i, data in enumerate(all_data):
        x = np.random.normal(i + 1, 0.05, size=len(data))
        ax.scatter(x, data, alpha=0.6, color=box_colours[i],
                  edgecolor='black', linewidth=0.5, s=30, zorder=3)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Total Boundary Cell Visits', fontsize=18)
    ax.set_title('Boundary Cell Incursions per Trial', fontsize=20)
    ax.grid(True, alpha=0.3, axis='y')

    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], alpha=0.7, label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], alpha=0.7, label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], alpha=0.7, label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'boundary_visits_boxplot')

In [ ]:
plot_boundary_visits()

In [ ]:
def plot_coverage_uniformity():
    configs = [
        ('No Mess',          'PhemBots_NoMess'),
        ('Consistent',       'PhemBots_CMess'),
        ('Random',           'PhemBots_Random_Mess'),
        ('No Mess',       'Auction_NoMess'),
        ('Consistent',    'Auction_CMess'),
        ('Random',        'Auction_Random_Mess'),
        ('Random',  'Boustrophedon_Random_Mess'),
    ]

    box_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]

    all_data = []
    labels = []

    for label, folder_key in configs:
        trials = load_trials(folder_key)
        trial_values = []
        for df in trials:
            grid = get_visit_grid(df)
            if grid is None:
                continue

            interior = grid[BORDER:-BORDER, BORDER:-BORDER].flatten()
            trial_values.append(interior.tolist())

        flat = [v for trial in trial_values for v in trial]
        all_data.append(flat)
        labels.append(label)

    fig, ax = plt.subplots(figsize=(12, 6))
    bp = ax.boxplot(all_data, patch_artist=True, notch=False,
                    showfliers=False,
                    medianprops=dict(color='black', linewidth=2))

    for patch, colour in zip(bp['boxes'], box_colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)

    for whisker in bp['whiskers']:
        whisker.set_color('black')
    for cap in bp['caps']:
        cap.set_color('black')

    for i, data in enumerate(all_data):
        if data:
            q1 = np.percentile(data, 25)
            q3 = np.percentile(data, 75)
            iqr = q3 - q1
            ax.text(i + 1, q3 + 0.5, f'IQR={iqr:.1f}',
                   ha='center', va='bottom', fontsize=18, color='black')

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Visit Count per Interior Cell', fontsize=18)
    ax.set_title('Coverage Uniformity - Distribution of Interior Cell Visit Counts\n'
                  'IQR = Interquartile Range', fontsize=18)
    ax.grid(True, alpha=0.3, axis='y')

    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], alpha=0.7, label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], alpha=0.7, label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], alpha=0.7, label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'coverage_uniformity_boxplot')

In [ ]:
plot_coverage_uniformity()

In [ ]:
def get_distance_traveled(df):
    """
    Calculate per-timestep distance traveled (mm) and cumulative distance
    from robot position data in snapshots.
    Returns arrays of (times, cumulative_distance, speed_mm_per_s)
    """
    snaps = get_snapshots(df)
    if snaps.empty:
        return None, None, None

    times = snaps['timestamp'].astype(float).values
    positions_raw = snaps['robot_positions'].values

    cell_w = 900.0 / 30.0 
    cell_h = 652.0 / 30.0 

    cumulative = [0.0]
    speeds = [0.0]

    for i in range(1, len(times)):
        try:
            prev_pos = ast.literal_eval(positions_raw[i-1])
            curr_pos = ast.literal_eval(positions_raw[i])
        except:
            cumulative.append(cumulative[-1])
            speeds.append(0.0)
            continue

        dt = times[i] - times[i-1]
        if dt <= 0:
            cumulative.append(cumulative[-1])
            speeds.append(0.0)
            continue

        total_dist = 0.0
        for robot_id in curr_pos:
            if robot_id in prev_pos:
                dr = curr_pos[robot_id][0] - prev_pos[robot_id][0]
                dc = curr_pos[robot_id][1] - prev_pos[robot_id][1]
                dist_mm = ((dr * cell_h)**2 + (dc * cell_w)**2)**0.5
                total_dist += dist_mm

        cumulative.append(cumulative[-1] + total_dist)
        speeds.append(total_dist / dt)

    return times, np.array(cumulative), np.array(speeds)

def plot_distance_traveled():
    conditions = [
        ('No Mess',         'PhemBots_NoMess',     'Auction_NoMess',     'Boustrophedon_Random_Mess'),
        ('Consistent Mess', 'PhemBots_CMess',       'Auction_CMess',      'Boustrophedon_Random_Mess'),
        ('Random Mess',     'PhemBots_Random_Mess', 'Auction_Random_Mess','Boustrophedon_Random_Mess'),
    ]

    system_keys = {
        'No Mess':         ['PhemBots_NoMess',     'Auction_NoMess',     'Boustrophedon_Random_Mess'],
        'Consistent Mess': ['PhemBots_CMess',       'Auction_CMess',      'Boustrophedon_Random_Mess'],
        'Random Mess':     ['PhemBots_Random_Mess', 'Auction_Random_Mess','Boustrophedon_Random_Mess'],
    }

    systems = [
        ('PhemBots',            SYSTEM_COLOURS['PhemBots']),
        ('Auction',                SYSTEM_COLOURS['Auction']),
        ('Boustrophedon (Random)', SYSTEM_COLOURS['Boustrophedon']),
    ]

    for cond_name, main_key, auction_key, boust_key in conditions:
        folder_keys = system_keys[cond_name]
        fig, ax = plt.subplots(figsize=(10, 6))

        for (sys_name, colour), folder_key in zip(systems, folder_keys):
            trials = load_trials(folder_key)
            if not trials:
                continue

            all_interp = []
            for df in trials:
                t, cum, spd = get_distance_traveled(df)
                if t is None:
                    continue
                all_interp.append(np.interp(INTERPOLATION_TIMES, t, cum))

            if not all_interp:
                continue

            mean_val = np.mean(all_interp, axis=0)
            std_val  = np.std(all_interp, axis=0)

            ax.errorbar(
                INTERPOLATION_TIMES, mean_val,
                yerr=std_val,
                fmt='o-', color=colour, alpha=0.8,
                markersize=2, linewidth=1.5,
                markevery=10, capsize=2,
                label=sys_name
            )

        if cond_name == 'Consistent Mess':
            time_counts = {}
            for t_mess, mess_colour in CONSISTENT_MESS_TIMES:
                count = time_counts.get(t_mess, 0)
                mc = MESS_COLOURS.get(mess_colour, 'gray')
                linestyle = (0, (5, 1)) if count == 0 else (0, (1, 1))
                ax.axvline(x=t_mess, color=mc, alpha=0.8, linewidth=1.5, linestyle=linestyle)
                time_counts[t_mess] = count + 1

        ax.set_xlabel('Time (s)', fontsize=18)
        ax.set_ylabel('Cumulative Distance (mm)', fontsize=18)
        ax.set_title(f'Cumulative Distance Traveled - {cond_name}', fontsize=20)
        ax.legend(fontsize=18)
        ax.set_xlim(0, TRIAL_DURATION)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_fig(fig, f'distance_cumulative_{cond_name.replace(" ","_").lower()}')

        fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

        for ax, (sys_name, colour), folder_key in zip(axes, systems, folder_keys):
            trials = load_trials(folder_key)
            if not trials:
                continue

            all_interp = []
            for df in trials:
                t, cum, spd = get_distance_traveled(df)
                if t is None:
                    continue
                all_interp.append(np.interp(INTERPOLATION_TIMES, t, spd))

            if not all_interp:
                continue

            mean_val = np.mean(all_interp, axis=0)
            std_val  = np.std(all_interp, axis=0)

            ax.errorbar(
                INTERPOLATION_TIMES, mean_val,
                yerr=std_val,
                fmt='o-', color=colour, alpha=0.8,
                markersize=2, linewidth=1.5,
                markevery=10, capsize=2,
                label=sys_name
            )

            if cond_name == 'Consistent Mess':
                time_counts = {}
                for t_mess, mess_colour in CONSISTENT_MESS_TIMES:
                    count = time_counts.get(t_mess, 0)
                    mc = MESS_COLOURS.get(mess_colour, 'gray')
                    linestyle = (0, (5, 1)) if count == 0 else (0, (1, 1))
                    ax.axvline(x=t_mess, color=mc, alpha=0.8,
                              linewidth=1.5, linestyle=linestyle)
                    time_counts[t_mess] = count + 1

            ax.set_ylabel('Speed (mm/s)', fontsize=18)
            ax.set_ylim(bottom=0)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=18, loc='upper left')

        all_max = max(ax.get_ylim()[1] for ax in axes)
        for ax in axes:
            ax.set_ylim(0, all_max)

        axes[-1].set_xlabel('Time (s)', fontsize=18)
        axes[-1].set_xlim(0, TRIAL_DURATION)
        fig.suptitle(f'Movement Speed - {cond_name}', fontsize=20)
        fig.tight_layout()
        save_fig(fig, f'distance_speed_{cond_name.replace(" ","_").lower()}')


In [ ]:

plot_distance_traveled()

In [ ]:
def plot_speed_histogram():
    conditions = [
        ('No Mess',         'PhemBots_NoMess',     'Auction_NoMess',     'Boustrophedon_Random_Mess'),
        ('Consistent Mess', 'PhemBots_CMess',       'Auction_CMess',      'Boustrophedon_Random_Mess'),
        ('Random Mess',     'PhemBots_Random_Mess', 'Auction_Random_Mess','Boustrophedon_Random_Mess'),
    ]

    for cond_name, main_key, auction_key, boust_key in conditions:
        fig, ax = plt.subplots(figsize=(10, 6))

        for sys_name, folder_key, colour in [
            ('PhemBots',            main_key,    SYSTEM_COLOURS['PhemBots']),
            ('Auction',                auction_key, SYSTEM_COLOURS['Auction']),
            ('Boustrophedon (Random)', boust_key,   SYSTEM_COLOURS['Boustrophedon']),
        ]:
            trials = load_trials(folder_key)
            if not trials:
                continue

            all_speeds = []
            for df in trials:
                t, cum, spd = get_distance_traveled(df)
                if spd is None:
                    continue
                all_speeds.extend(spd.tolist())

            if not all_speeds:
                continue

            ax.hist(all_speeds, bins=50, color=colour, alpha=0.5,
                   density=True, label=sys_name)

        ax.set_xlabel('Speed (mm/s)', fontsize=18)
        ax.set_ylabel('Density', fontsize=18)
        ax.set_title(f'Movement Speed Distribution - {cond_name}', fontsize=20)
        ax.legend(fontsize=18)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_fig(fig, f'speed_histogram_{cond_name.replace(" ","_").lower()}')

In [ ]:
plot_speed_histogram()

In [ ]:
def plot_speed_boxplot():
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',       'Auction_NoMess'),
        ('Auction\nConsistent',    'Auction_CMess'),
        ('Auction\nRandom',        'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',  'Boustrophedon_Random_Mess'),
    ]

    box_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]

    all_data = []
    labels = []

    for label, folder_key in configs:
        trials = load_trials(folder_key)
        all_speeds = []
        for df in trials:
            t, cum, spd = get_distance_traveled(df)
            if spd is None:
                continue
            all_speeds.extend(spd.tolist())
        all_data.append(all_speeds)
        labels.append(label)

    fig, ax = plt.subplots(figsize=(12, 6))
    bp = ax.boxplot(all_data, patch_artist=True, notch=False,
                   showfliers=False,
                   medianprops=dict(color='black', linewidth=2))

    for patch, colour in zip(bp['boxes'], box_colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)

    for whisker in bp['whiskers']:
        whisker.set_color('black')
    for cap in bp['caps']:
        cap.set_color('black')

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Speed (mm/s)', fontsize=18)
    ax.set_title('Movement Speed Distribution', fontsize=20)
    ax.grid(True, alpha=0.3, axis='y')

    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], alpha=0.7, label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], alpha=0.7, label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], alpha=0.7, label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'speed_boxplot_combined')

In [ ]:
plot_speed_boxplot()

In [ ]:
def plot_collision_resolution():
    """
    How quickly collisions are resolved once they start.
    Bar chart of mean collision duration per system/condition.
    """
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',       'Auction_NoMess'),
        ('Auction\nConsistent',    'Auction_CMess'),
        ('Auction\nRandom',        'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',  'Boustrophedon_Random_Mess'),
    ]
 
    bar_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]
 
    means, stds, labels = [], [], []
 
    for label, folder_key in configs:
        trials = load_trials(folder_key)
        all_durations = []
 
        for df in trials:
            snaps = get_snapshots(df)
            if snaps.empty:
                continue
            times = snaps['timestamp'].astype(float).values
            collisions = snaps['near_collisions'].astype(float).values
 
            in_collision = False
            collision_start = None
            for t, c in zip(times, collisions):
                if c > 0 and not in_collision:
                    in_collision = True
                    collision_start = t
                elif c == 0 and in_collision:
                    all_durations.append(t - collision_start)
                    in_collision = False
 
        if all_durations:
            means.append(np.mean(all_durations))
            stds.append(np.std(all_durations))
        else:
            means.append(0)
            stds.append(0)
        labels.append(label)
 
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=stds, capsize=5, color=bar_colours,
           alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Mean Collision Duration (s)', fontsize=18)
    ax.set_title('Mean Collision Resolution Time per System/Condition\n'
                 '(PhemBots collisions during mess response are intentional)', fontsize=18)
    ax.grid(True, alpha=0.3, axis='y')
 
    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'collision_resolution_bar')

In [ ]:
plot_collision_resolution()

In [ ]:
def plot_total_reward_bar():
    """Bar chart of total reward at end of trial for each system/condition."""
    configs = [
        ('PhemBots\nNo Mess',          'PhemBots_NoMess'),
        ('PhemBots\nConsistent',       'PhemBots_CMess'),
        ('PhemBots\nRandom',           'PhemBots_Random_Mess'),
        ('Auction\nNo Mess',       'Auction_NoMess'),
        ('Auction\nConsistent',    'Auction_CMess'),
        ('Auction\nRandom',        'Auction_Random_Mess'),
        ('Boustrophedon\nRandom',  'Boustrophedon_Random_Mess'),
    ]
 
    bar_colours = [
        SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'], SYSTEM_COLOURS['PhemBots'],
        SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'], SYSTEM_COLOURS['Auction'],
        SYSTEM_COLOURS['Boustrophedon'],
    ]
 
    means, stds, labels = [], [], []
 
    for label, folder_key in configs:
        trials = load_trials(folder_key)
        values = []
        for df in trials:
            snaps = get_snapshots(df)
            if snaps.empty or 'total_reward' not in snaps.columns:
                continue
            try:
                values.append(float(snaps['total_reward'].dropna().iloc[-1]))
            except:
                continue
        if values:
            means.append(np.mean(values))
            stds.append(np.std(values))
        else:
            means.append(0)
            stds.append(0)
        labels.append(label)
 
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=stds, capsize=5, color=bar_colours,
           alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel('Total Reward Collected', fontsize=18)
    ax.set_title('Total Reward at End of Trial per System/Condition', fontsize=20)
    ax.grid(True, alpha=0.3, axis='y')
 
    legend_elements = [
        mpatches.Patch(color=SYSTEM_COLOURS['PhemBots'], label='PhemBots'),
        mpatches.Patch(color=SYSTEM_COLOURS['Auction'], label='Auction'),
        mpatches.Patch(color=SYSTEM_COLOURS['Boustrophedon'], label='Boustrophedon'),
    ]
    ax.legend(handles=legend_elements, fontsize=18)
    fig.tight_layout()
    save_fig(fig, 'total_reward_bar')

In [ ]:
plot_total_reward_bar()